In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, fixed

import pmagpy.pmag as pmag
import pmagpy.pmagplotlib as pmagplotlib
import pmagpy.ipmag as ipmag

from dtw import dtw
from scipy import interpolate

-W- cartopy is not installed
    If you want to make maps, install using conda:
    conda install cartopy
Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
def parse_data(df, row_name,header_length = 1):
    data = []

    for i, row in df.iterrows():
        if i+1 > header_length:
            data.append([i,row[row_name]])

    return np.asarray(data)    

In [3]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

In [4]:
def weighted_nan_dist(a, b, weights=None, scales=None, nan_penalty=1.0):
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    n = len(a)

    if weights is None:
        weights = np.ones(n)

    if scales is None:
        scales = np.ones(n)

    weights = np.asarray(weights, dtype=float)
    scales = np.asarray(scales, dtype=float)

    total = 0.0

    for ak, bk, wk, sk in zip(a, b, weights, scales):
        if sk == 0 or np.isnan(sk):
            sk = 1.0

        if np.isnan(ak) or np.isnan(bk):
            total += wk * nan_penalty
        else:
            total += wk * abs(ak - bk) / sk

    return total

In [5]:
def anchored_dtw_multi(x, y, anchors, weights=None, scales=None,
                       nan_penalty=1.0, **dtw_kwargs):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if x.ndim == 1:
        x = x.reshape(-1, 1)

    if y.ndim == 1:
        y = y.reshape(-1, 1)

    anchors = sorted(anchors)

    def dist_method(a, b):
        return weighted_nan_dist(
            a, b,
            weights=weights,
            scales=scales,
            nan_penalty=nan_penalty
        )

    full_index1 = []
    full_index2 = []
    total_distance = 0.0

    for (i_start, j_start), (i_end, j_end) in zip(anchors[:-1], anchors[1:]):
        x_seg = x[i_start:i_end + 1]
        y_seg = y[j_start:j_end + 1]

        alignment = dtw(
            x_seg,
            y_seg,
            dist_method=dist_method,
            keep_internals=True,
            **dtw_kwargs
        )

        idx1 = np.array(alignment.index1) + i_start
        idx2 = np.array(alignment.index2) + j_start

        if len(full_index1) > 0:
            idx1 = idx1[1:]
            idx2 = idx2[1:]

        full_index1.extend(idx1)
        full_index2.extend(idx2)
        total_distance += alignment.distance

    return np.array(full_index1), np.array(full_index2), total_distance

In [6]:
def dynamic_draw(values, x_pos):

    values = np.asarray(values, dtype=float)

    point_y = values[x_pos]

    fig, ax = plt.subplots()
    ax.plot(values)

    if not np.isnan(point_y):
        ax.scatter(x_pos, point_y, label=f'{x_pos}')
        ax.legend(loc='upper right')
    else:
        ax.legend('No valid data',loc='upper right')
        
    plt.show()

In [7]:
def original_points_on_warped_axis(idx):
    """
    For each original point, find where it appears on the warped DTW axis.
    If one original point is repeated by DTW stretching, place the marker
    at the mean position of that repeated segment.
    """
    idx = np.asarray(idx)

    original_ids = np.unique(idx)
    warped_positions = np.array([
        np.mean(np.flatnonzero(idx == original_id))
        for original_id in original_ids
    ])

    return original_ids, warped_positions

Please enter your file name here.

In [9]:
xl_file = pd.ExcelFile('I_full data.xlsx')

dfs = {sheet_name: xl_file.parse(sheet_name) 
          for sheet_name in xl_file.sheet_names}

Please enter output file

In [10]:
output_file_name = "output.xlsx"

Please ender max and min depth

In [11]:
depth_min = 0
depth_max = 120

These are the columns of your file.

In [12]:
dfs['B'].columns

Index(['thickness, mm', 'Piece', 'sample №', 'thick, mm', 'weight, g',
       'Vol, cm3', 'depth, mm', 'Dgeo', 'Igeo', 'MADgeo', 'NRMmax (Am2/kg)',
       'ARMmax (Am2/kg)'],
      dtype='str')

Please enter desired sheets

In [17]:
name1 = 'A'
name2 = 'B'

In [18]:
sheet_1 = dfs[name1]
sheet_2 = dfs[name2]

In [19]:
sample_1 = parse_data(sheet_1, 'Igeo', 1)
sample_2 = parse_data(sheet_2, 'Igeo', 1)

In [20]:
interact(
    dynamic_draw,
    #idx = fixed(sample_1[:,0]),
    values = fixed(sample_1[:,1]),
    x_pos = widgets.IntSlider(min = 0, max = sample_1[:,0].shape[0] - 1)
)

interactive(children=(IntSlider(value=0, description='x_pos', max=36), Output()), _dom_classes=('widget-intera…

<function __main__.dynamic_draw(values, x_pos)>

In [21]:
interact(
    dynamic_draw,
    #idx = fixed(sample_2[:,0]),
    values = fixed(sample_2[:,1]),
    x_pos = widgets.IntSlider(min = 0, max = sample_2[:,0].shape[0] - 1)
)

interactive(children=(IntSlider(value=0, description='x_pos', max=36), Output()), _dom_classes=('widget-intera…

<function __main__.dynamic_draw(values, x_pos)>

Enter anchors

In [ ]:
anchors = [
    (0, 0),
    (4, 3),
    (15, 13),
    (29, 29),
    (len(sample_1[:, 1]) - 1, len(sample_2[:, 1]) - 1)
]

Enter desiered params

In [ ]:
params = [
    "Dgeo",
    "Igeo",
    "NRMmax (Am2/kg)",
    "ARMmax (Am2/kg)"
]

skip1 = 2
skip2 = 3

x = sheet_1[params].iloc[skip1:].to_numpy(dtype=float)
y = sheet_2[params].iloc[skip2:].to_numpy(dtype=float)

combined = np.vstack([x, y])
scales = np.nanstd(combined, axis=0)

weights = np.array([
    0.1,  # Dgeo
    1.0,  # Igeo
    0.1,  # NRMmax
    1.0   # ARMmax
])

idx1, idx2, dist = anchored_dtw_multi(
    x,
    y,
    anchors,
    weights=weights,
    scales=scales,
    nan_penalty=1.0,
    step_pattern="symmetric2"
)

In [ ]:
param_name = "Igeo"
param_id = params.index(param_name)

warp_axis = np.arange(len(idx1))

x_warped = x[idx1, param_id]
y_warped = y[idx2, param_id]

orig_x_ids, x_point_positions = original_points_on_warped_axis(idx1)
orig_y_ids, y_point_positions = original_points_on_warped_axis(idx2)

plt.figure(figsize=(10, 5))

plt.plot(warp_axis, x_warped, label=f"sample 1 warped: {param_name}")
plt.plot(warp_axis, y_warped, label=f"sample 2 warped: {param_name}")

plt.scatter(
    x_point_positions,
    x[orig_x_ids, param_id],
    marker="o",
    label="sample 1 original point locations"
)

plt.scatter(
    y_point_positions,
    y[orig_y_ids, param_id],
    marker="x",
    label="sample 2 original point locations"
)

inc = x[idx1, param_id]
inc2 = y[idx2, param_id]
plt.xlabel("DTW warped coordinate")
plt.ylabel(param_name)
plt.legend(loc = 'upper right')
plt.show()

In [ ]:
param_name = "Dgeo"
param_id = params.index(param_name)

warp_axis = np.arange(len(idx1))

x_warped = x[idx1, param_id]
y_warped = y[idx2, param_id]

orig_x_ids, x_point_positions = original_points_on_warped_axis(idx1)
orig_y_ids, y_point_positions = original_points_on_warped_axis(idx2)

plt.figure(figsize=(10, 5))

plt.plot(warp_axis, x_warped, label=f"sample 1 warped: {param_name}")
plt.plot(warp_axis, y_warped, label=f"sample 2 warped: {param_name}")

plt.scatter(
    x_point_positions,
    x[orig_x_ids, param_id],
    marker="o",
    label="sample 1 original point locations"
)

plt.scatter(
    y_point_positions,
    y[orig_y_ids, param_id],
    marker="x",
    label="sample 2 original point locations"
)

dec = x[idx1, param_id]
dec2 = y[idx2, param_id]

plt.xlabel("DTW warped coordinate")
plt.ylabel(param_name)
plt.legend(loc = 'upper right')
plt.show()

In [ ]:
param_name = "ARMmax (Am2/kg)"
param_id = params.index(param_name)

plt.figure(figsize=(10, 5))

plt.plot(x[idx1, param_id], label=f"sample 1 warped: {param_name}")
plt.plot(y[idx2, param_id], label=f"sample 2 warped: {param_name}")

plt.legend()
plt.show()

In [ ]:
param_name = "NRMmax (Am2/kg)"
param_id = params.index(param_name)

plt.figure(figsize=(10, 5))

plt.plot(x[idx1, param_id], label=f"sample 1 warped: {param_name}")
plt.plot(y[idx2, param_id], label=f"sample 2 warped: {param_name}")

plt.legend()
plt.show()

In [ ]:
ipmag.plot_net(1)
ipmag.plot_di(dec=dec,inc=inc,color='blue',edge='black')
ipmag.plot_di(dec=dec2,inc=inc2,color='red',edge='black')